In [12]:
import os
import time
import logging
# from mapbox import Geocoder
import pandas as pd
from shapely.geometry import Point
from tqdm.auto import tqdm
import geopandas as gpd
import numpy as np
from scipy import stats
import osmnx as ox
import geopandas as gpd
import numpy as np

In [51]:
cian_buy = pd.read_csv('cian_buy_builds_23-03_2026.csv')
cian_rent = pd.read_csv('cian_rent_builds_11-01_2026.csv')
kaggle_buy = pd.read_csv('moscow_builds_2021_price.csv').drop('house_id', axis=1)
domclick_incomes = pd.read_csv('domclick_incomes.csv')

cian_buy['year']=2026
cian_rent['year']=2026
kaggle_buy['year']=2021
domclick_incomes['year']=2025

cian_buy['type']='buy'
cian_rent['type']='rent'
kaggle_buy['type']='buy'
domclick_incomes['type']='incomes'

In [52]:
data = pd.concat([cian_buy, cian_rent, kaggle_buy, domclick_incomes], axis= 0)
data

,lat,lon,price_per_m2,building_type,object_type,levels,ads_count,year,type,address,...,between_45_54_num,between_55_64_num,over_65_num,men_num,women_num,with_children_num,with_pets_num,with_car_num,median_income_pop,avg_hcs_cost_pop
0,55.211456,37.071235,151093.643199,0.0,1.0,5.0,2.0,2026,buy,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,55.212283,37.072160,113981.762918,1.0,1.0,5.0,1.0,2026,buy,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,55.213315,37.076095,107657.232704,0.0,1.0,2.0,1.0,2026,buy,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,55.214039,37.071513,110294.117647,3.0,1.0,5.0,1.0,2026,buy,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,55.214270,37.067543,131608.432847,3.0,1.0,4.0,2.0,2026,buy,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9083,55.725257,37.675679,NaN,NaN,NaN,NaN,NaN,2025,incomes,"Россия, Москва, улица Мельникова, 3 к5",...,69.534636,39.867482,15.695576,199.085995,160.080512,94.568541,67.631053,193.590747,5.296126e+07,4.136880e+06
9084,55.678550,37.657626,NaN,NaN,NaN,NaN,NaN,2025,incomes,"Россия, Москва, Нагатинская улица, 29 к2",...,21.838233,14.936326,6.816435,52.854079,53.986590,28.184569,21.197189,41.251183,1.270549e+07,1.185077e+06
9085,55.715319,37.898570,NaN,NaN,NaN,NaN,NaN,2025,incomes,"Лухмановская улица, 5",...,68.316198,27.169355,12.766323,173.785670,153.555957,107.662661,57.022911,136.632395,3.117307e+07,3.461310e+06
9086,55.658937,37.519565,NaN,NaN,NaN,NaN,NaN,2025,incomes,"улица Обручева, 19 к3",...,87.769838,36.152155,16.375265,321.058031,74.479768,113.400687,63.642032,372.359284,4.294987e+07,3.459769e+06


In [53]:
data.columns

Index(['lat', 'lon', 'price_per_m2', 'building_type', 'object_type', 'levels',
       'ads_count', 'year', 'type', 'address', 'total_apartments',
       'inhab_count', 'under_24_num', 'between_25_34_num', 'between_35_44_num',
       'between_45_54_num', 'between_55_64_num', 'over_65_num', 'men_num',
       'women_num', 'with_children_num', 'with_pets_num', 'with_car_num',
       'median_income_pop', 'avg_hcs_cost_pop'],
      dtype='object')

In [ ]:
'year' и 'type' - столбцы, для которых должны быть выполнено условие, а также префиксы. Например, 'rent_2026_price_per_m2'
1. Данные аренды Циан 2026
'year'==2026
'type'=='rent'

'price_per_m2' - среднее, медиана,
'ads_count' - сумма

2. Данные продажи Циан 2026
'year'==2026
'type'=='buy'

'price_per_m2' - среднее, медиана,
'building_type' - самое распространенное значение
'object_type' - среднее (то есть средняя доля вторичного жилья)
'ads_count' - сумма

3. Данные продажи kaggle 2021
'year'==2021
'type'=='buy'

'price_per_m2' - среднее, медиана,
'building_type' - самое распространенное значение
'object_type' - среднее (то есть средняя доля вторичного жилья)
'ads_count' - сумма

4. Доходы и соц.дем. Домклик 2025
'year'==2025
'type'=='incomes'

'under_24_num' - сумма,
'between_25_34_num' - сумма,
'between_35_44_num' - сумма,
'between_45_54_num' - сумма,
'between_55_64_num' - сумма,
'over_65_num' - сумма,
'men_num' - сумма,
'women_num' - сумма,
'with_children_num' - сумма,
'with_pets_num' - сумма,
'with_car_num' - сумма,
'median_income_pop' - среднее, медиана, 
'avg_hcs_cost_pop' - среднее, медиана,

### Изохрона 15 минут пешком

In [54]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами
WALK_SPEED_KMH = 4.5
METERS_PER_MIN = WALK_SPEED_KMH * 1000 / 60
WALK_TIME_MIN = 15

# --- ПРАВИЛА АГРЕГАЦИИ ---
# Здесь мы задаем фильтры и нужные метрики для каждой группы
AGG_RULES = {
    'rent_2026': {
        'mask': lambda df: (df['year'] == 2026) & (df['type'] == 'rent'),
        'metrics': {'price_per_m2': ['mean', 'median'], 'ads_count': ['sum']}
    },
    'buy_2026': {
        'mask': lambda df: (df['year'] == 2026) & (df['type'] == 'buy'),
        'metrics': {'price_per_m2': ['mean', 'median'], 'building_type': ['mode'], 'object_type': ['mean'], 'ads_count': ['sum']}
    },
    'buy_2021': {
        'mask': lambda df: (df['year'] == 2021) & (df['type'] == 'buy'),
        'metrics': {'price_per_m2': ['mean', 'median'], 'building_type': ['mode'], 'object_type': ['mean'], 'ads_count': ['sum']}
    },
    'incomes_2025': {
        'mask': lambda df: (df['year'] == 2025) & (df['type'] == 'incomes'),
        'metrics': {
            'under_24_num': ['sum'], 'between_25_34_num': ['sum'], 'between_35_44_num': ['sum'],
            'between_45_54_num': ['sum'], 'between_55_64_num': ['sum'], 'over_65_num': ['sum'],
            'men_num': ['sum'], 'women_num': ['sum'], 'with_children_num': ['sum'],
            'with_pets_num': ['sum'], 'with_car_num': ['sum'],
            'median_income_pop': ['mean', 'median'], 'avg_hcs_cost_pop': ['mean', 'median']
        }
    }
}

# Предварительно генерируем словарь со всеми NaN для пустых изохрон или ошибок
ALL_EMPTY_METRICS = {}
for prefix, config in AGG_RULES.items():
    for col, funcs in config['metrics'].items():
        for func in funcs:
            ALL_EMPTY_METRICS[f"{prefix}_{col}_{func}"] = np.nan

# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
# G_walk = ox.load_graphml(f'{GRAPH_PATH}/moscow_walk.graphml')
# G_walk = ox.project_graph(G_walk) # Проекция в метры (UTM)
current_crs = G_walk.graph['crs']

for u, v, k, edge_data in G_walk.edges(keys=True, data=True):
    length = edge_data.get('length')
    if length is not None:
        edge_data['time'] = float(length) / METERS_PER_MIN

# --- ПОДГОТОВКА НОВЫХ ДАННЫХ (data) ---
# Предполагается, что переменная data уже загружена как DataFrame
# data = pd.read_csv('your_data_file.csv')

print("Подготовка пространственных данных...")
data['geometry'] = gpd.points_from_xy(data.lon, data.lat)
gdf_data = gpd.GeoDataFrame(data, geometry='geometry', crs="EPSG:4326")

# Проецируем данные в ту же систему координат, что и граф
gdf_data = gdf_data.to_crs(current_crs)

# --- НАЧАЛО ОБРАБОТКИ ---
print('НАЧАЛО ОБРАБОТКИ')
fitness = pd.read_csv('data/fitness_w_centerd.csv')
results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0:
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_walk, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_walk, center_node, radius=WALK_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            results.append({'fitness_id': idx, **ALL_EMPTY_METRICS})
            continue

        poly = gpd.GeoSeries([Point(node_data['x'], node_data['y']) for _, node_data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ С НОВЫМ ДАТАФРЕЙМОМ ---
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_data, how='inner', predicate='intersects')
        
        if hits.empty:
            results.append({'fitness_id': idx, **ALL_EMPTY_METRICS})
            continue

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {'fitness_id': idx}
        
        for prefix, config in AGG_RULES.items():
            # Фильтруем данные под конкретную группу (например, аренда 2026)
            subset = hits[config['mask'](hits)]
            is_empty = subset.empty
            
            for col, funcs in config['metrics'].items():
                for func in funcs:
                    col_name = f"{prefix}_{col}_{func}"
                    
                    # Если данных нет или весь столбец пустой
                    if is_empty or subset[col].dropna().empty:
                        row_result[col_name] = np.nan
                        continue
                        
                    # Агрегация
                    if func == 'mean':
                        row_result[col_name] = subset[col].mean()
                    elif func == 'median':
                        row_result[col_name] = subset[col].median()
                    elif func == 'sum':
                        row_result[col_name] = subset[col].sum()
                    elif func == 'mode':
                        mode_series = subset[col].mode()
                        row_result[col_name] = mode_series.iloc[0] if not mode_series.empty else np.nan

        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        results.append({'fitness_id': idx, **ALL_EMPTY_METRICS})

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df = pd.DataFrame(results)
print(f"Обработка завершена за {time.time() - start:.2f} секунд.")
# res_df.to_csv('your_results.csv', index=False)
res_df.head()

Загружаем пешеходный граф Москвы...
Подготовка пространственных данных...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Обрабатываем объект 251/5029...
Обрабатываем объект 501/5029...
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Обрабатываем объект 1751/5029...
Обрабатываем объект 2001/5029...
Обрабатываем объект 2251/5029...
Обрабатываем объект 2501/5029...
Обрабатываем объект 2751/5029...
Обрабатываем объект 3001/5029...
Обрабатываем объект 3251/5029...
Обрабатываем объект 3501/5029...
Обрабатываем объект 3751/5029...
Обрабатываем объект 4001/5029...
Обрабатываем объект 4251/5029...
Обрабатываем объект 4501/5029...
Обрабатываем объект 4751/5029...
Обрабатываем объект 5001/5029...
Обработка завершена за 17353.23 секунд.


,fitness_id,rent_2026_price_per_m2_mean,rent_2026_price_per_m2_median,rent_2026_ads_count_sum,buy_2026_price_per_m2_mean,buy_2026_price_per_m2_median,buy_2026_building_type_mode,buy_2026_object_type_mean,buy_2026_ads_count_sum,buy_2021_price_per_m2_mean,...,incomes_2025_over_65_num_sum,incomes_2025_men_num_sum,incomes_2025_women_num_sum,incomes_2025_with_children_num_sum,incomes_2025_with_pets_num_sum,incomes_2025_with_car_num_sum,incomes_2025_median_income_pop_mean,incomes_2025_median_income_pop_median,incomes_2025_avg_hcs_cost_pop_mean,incomes_2025_avg_hcs_cost_pop_median
0,0,2569.032714,2400.529101,178.0,7.970341e+05,6.851851e+05,3.0,1.0,310.0,484958.830406,...,293.768831,4479.595286,4299.158303,1764.488356,1659.387075,2936.749763,1.598056e+07,1.139869e+07,1.419180e+06,1.014636e+06
1,1,2665.939836,2429.929930,133.0,6.126966e+05,5.592359e+05,3.0,1.0,293.0,381390.691083,...,518.605402,5250.071598,4833.551847,2199.731608,1932.711980,3764.449188,3.588802e+07,2.596099e+07,2.802393e+06,2.171438e+06
2,2,2927.356141,2927.356141,5.0,1.676609e+06,1.539448e+06,2.0,1.0,24.0,458417.041057,...,18.306000,87.606000,92.394000,40.482000,38.646000,88.740000,2.906982e+07,2.906982e+07,2.493000e+06,2.493000e+06
3,3,1599.444201,1563.227335,131.0,3.533763e+05,3.500118e+05,1.0,1.0,443.0,219982.768485,...,451.194917,7665.148883,6783.487002,4272.857678,2776.274080,5478.692728,5.564087e+07,4.764006e+07,5.264354e+06,4.647412e+06
4,4,2146.670277,1854.935520,23.0,7.206844e+05,7.276922e+05,2.0,1.0,594.0,343284.409889,...,57.500513,1994.414908,1195.160212,687.072113,518.935036,1051.999287,4.762166e+07,2.725802e+07,4.757979e+06,3.155405e+06


In [55]:
res_df = pd.DataFrame(results)
print("Готово!")
end = time.time()
print(f"Время: {end - start:.4f} сек")


for col in list(res_df.columns[1:]):
    old_name = col
    new_name = col + '_15ped'
    res_df = res_df.rename(columns={old_name: new_name})
    
fitness_iso_realestate_ped = pd.concat([fitness,res_df[1:]], axis=1)
fitness_iso_realestate_ped.to_csv('fitness_realestate_iso_15ped.csv', index = False)
fitness_iso_realestate_ped.to_excel('fitness_realestate_iso_15ped.xlsx', index = False)

Готово!
Время: 17353.3422 сек


## Изохрона 15 мин на машине

In [56]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами

DRIVE_SPEED_KMH = 30 # Средняя скорость авто в городе
METERS_PER_MIN_DRIVE = DRIVE_SPEED_KMH * 1000 / 60
DRIVE_TIME_MIN = 15
iso_type = 'drive'
MAX_DRIVE_METERS = DRIVE_TIME_MIN * METERS_PER_MIN_DRIVE

# --- ПРАВИЛА АГРЕГАЦИИ ---
# Здесь мы задаем фильтры и нужные метрики для каждой группы
AGG_RULES = {
    'rent_2026': {
        'mask': lambda df: (df['year'] == 2026) & (df['type'] == 'rent'),
        'metrics': {'price_per_m2': ['mean', 'median'], 'ads_count': ['sum']}
    },
    'buy_2026': {
        'mask': lambda df: (df['year'] == 2026) & (df['type'] == 'buy'),
        'metrics': {'price_per_m2': ['mean', 'median'], 'building_type': ['mode'], 'object_type': ['mean'], 'ads_count': ['sum']}
    },
    'buy_2021': {
        'mask': lambda df: (df['year'] == 2021) & (df['type'] == 'buy'),
        'metrics': {'price_per_m2': ['mean', 'median'], 'building_type': ['mode'], 'object_type': ['mean'], 'ads_count': ['sum']}
    },
    'incomes_2025': {
        'mask': lambda df: (df['year'] == 2025) & (df['type'] == 'incomes'),
        'metrics': {
            'under_24_num': ['sum'], 'between_25_34_num': ['sum'], 'between_35_44_num': ['sum'],
            'between_45_54_num': ['sum'], 'between_55_64_num': ['sum'], 'over_65_num': ['sum'],
            'men_num': ['sum'], 'women_num': ['sum'], 'with_children_num': ['sum'],
            'with_pets_num': ['sum'], 'with_car_num': ['sum'],
            'median_income_pop': ['mean', 'median'], 'avg_hcs_cost_pop': ['mean', 'median']
        }
    }
}

# Предварительно генерируем словарь со всеми NaN для пустых изохрон или ошибок
ALL_EMPTY_METRICS = {}
for prefix, config in AGG_RULES.items():
    for col, funcs in config['metrics'].items():
        for func in funcs:
            ALL_EMPTY_METRICS[f"{prefix}_{col}_{func}"] = np.nan

# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
# --- ЗАГРУЗКА ГРАФА (ОДИН РАЗ) ---
print("Загружаем дорожный граф Москвы...")
G_drive = ox.load_graphml(f'{GRAPH_PATH}/moscow_drive.graphml')
G_drive = ox.project_graph(G_drive) # Проекция в метры (UTM)
current_crs = G_drive.graph['crs']

for u, v, k, edge_data in G_drive.edges(keys=True, data=True):
    length = edge_data.get('length')
    if length is not None:
        edge_data['time'] = float(length) / METERS_PER_MIN_DRIVE

# --- ПОДГОТОВКА НОВЫХ ДАННЫХ (data) ---
# Предполагается, что переменная data уже загружена как DataFrame
# data = pd.read_csv('your_data_file.csv')

print("Подготовка пространственных данных...")
data['geometry'] = gpd.points_from_xy(data.lon, data.lat)
gdf_data = gpd.GeoDataFrame(data, geometry='geometry', crs="EPSG:4326")

# Проецируем данные в ту же систему координат, что и граф
gdf_data = gdf_data.to_crs(current_crs)

# --- НАЧАЛО ОБРАБОТКИ ---
print('НАЧАЛО ОБРАБОТКИ')
fitness = pd.read_csv('data/fitness_w_centerd.csv')
results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0:
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_drive, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_drive, center_node, radius=DRIVE_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            results.append({'fitness_id': idx, **ALL_EMPTY_METRICS})
            continue

        poly = gpd.GeoSeries([Point(node_data['x'], node_data['y']) for _, node_data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ С НОВЫМ ДАТАФРЕЙМОМ ---
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_data, how='inner', predicate='intersects')
        
        if hits.empty:
            results.append({'fitness_id': idx, **ALL_EMPTY_METRICS})
            continue

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {'fitness_id': idx}
        
        for prefix, config in AGG_RULES.items():
            # Фильтруем данные под конкретную группу (например, аренда 2026)
            subset = hits[config['mask'](hits)]
            is_empty = subset.empty
            
            for col, funcs in config['metrics'].items():
                for func in funcs:
                    col_name = f"{prefix}_{col}_{func}"
                    
                    # Если данных нет или весь столбец пустой
                    if is_empty or subset[col].dropna().empty:
                        row_result[col_name] = np.nan
                        continue
                        
                    # Агрегация
                    if func == 'mean':
                        row_result[col_name] = subset[col].mean()
                    elif func == 'median':
                        row_result[col_name] = subset[col].median()
                    elif func == 'sum':
                        row_result[col_name] = subset[col].sum()
                    elif func == 'mode':
                        mode_series = subset[col].mode()
                        row_result[col_name] = mode_series.iloc[0] if not mode_series.empty else np.nan

        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        results.append({'fitness_id': idx, **ALL_EMPTY_METRICS})

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df2 = pd.DataFrame(results)
print(f"Обработка завершена за {time.time() - start:.2f} секунд.")
# res_df.to_csv('your_results.csv', index=False)
res_df2.head()

Загружаем пешеходный граф Москвы...
Загружаем дорожный граф Москвы...
Подготовка пространственных данных...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Обрабатываем объект 251/5029...
Обрабатываем объект 501/5029...
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Обрабатываем объект 1751/5029...
Обрабатываем объект 2001/5029...
Обрабатываем объект 2251/5029...
Обрабатываем объект 2501/5029...
Обрабатываем объект 2751/5029...
Обрабатываем объект 3001/5029...
Обрабатываем объект 3251/5029...
Обрабатываем объект 3501/5029...
Обрабатываем объект 3751/5029...
Обрабатываем объект 4001/5029...
Обрабатываем объект 4251/5029...
Обрабатываем объект 4501/5029...
Обрабатываем объект 4751/5029...
Обрабатываем объект 5001/5029...
Обработка завершена за 1659.21 секунд.


,fitness_id,rent_2026_price_per_m2_mean,rent_2026_price_per_m2_median,rent_2026_ads_count_sum,buy_2026_price_per_m2_mean,buy_2026_price_per_m2_median,buy_2026_building_type_mode,buy_2026_object_type_mean,buy_2026_ads_count_sum,buy_2021_price_per_m2_mean,...,incomes_2025_over_65_num_sum,incomes_2025_men_num_sum,incomes_2025_women_num_sum,incomes_2025_with_children_num_sum,incomes_2025_with_pets_num_sum,incomes_2025_with_car_num_sum,incomes_2025_median_income_pop_mean,incomes_2025_median_income_pop_median,incomes_2025_avg_hcs_cost_pop_mean,incomes_2025_avg_hcs_cost_pop_median
0,0,2509.382185,2200.000000,4646.0,718009.753273,569526.627219,3.0,1.0,11352.0,419629.563536,...,20954.247793,256743.489536,217356.518119,112894.313981,95006.381538,195321.458886,3.608168e+07,2.271735e+07,3.152848e+06,2.023261e+06
1,1,2501.949936,2192.982456,5113.0,708760.726765,569299.476045,3.0,1.0,11993.0,414848.102537,...,22819.914865,266176.706363,232728.496507,124393.695362,101675.262556,211092.343983,4.216909e+07,2.306198e+07,3.675237e+06,2.053850e+06
2,2,2749.337227,2368.421053,4109.0,865945.056155,665859.564165,3.0,1.0,10268.0,484935.848332,...,19980.787163,205679.870055,182477.079705,99085.859371,79465.424893,177832.611613,4.489302e+07,2.495509e+07,4.005343e+06,2.180630e+06
3,3,1449.723662,1388.888889,739.0,327670.406543,329476.987143,2.0,1.0,2147.0,285557.417886,...,5607.509861,79887.819111,71518.433520,47124.881442,31577.360133,66209.520159,6.530275e+07,5.087054e+07,5.803214e+06,4.522334e+06
4,4,2618.386377,2290.525751,4008.0,762641.064328,585412.462704,3.0,1.0,10820.0,435490.283195,...,18053.546619,243762.579731,196575.205915,105122.692396,85734.924727,185035.332030,4.169766e+07,2.419067e+07,3.601425e+06,2.103061e+06


In [60]:
res_df2 = pd.DataFrame(results)
print("Готово!")
# end = time.time()
print(f"Время: {end - start:.4f} сек")


for col in list(res_df2.columns[1:]):
    old_name = col
    new_name = col + '_15drive'
    res_df2 = res_df2.rename(columns={old_name: new_name})
    
fitness_iso_realestate_drive = pd.concat([fitness,res_df2[1:]], axis=1)
fitness_iso_realestate_drive.to_csv('fitness_realestate_iso_15drive.csv', index = False)
fitness_iso_realestate_drive.to_excel('fitness_realestate_iso_15drive.xlsx', index = False)

fitness_iso_realestate_ped_drive = pd.concat([fitness_iso_realestate_ped,res_df2[1:]], axis=1)
fitness_iso_realestate_ped_drive.to_csv('fitness_realestate_iso_15ped_drive.csv', index = False)
fitness_iso_realestate_ped_drive.to_excel('fitness_realestate_iso_15ped_drive.xlsx', index = False)

Готово!
Время: -27.9417 сек


In [61]:
res_df2

,fitness_id,rent_2026_price_per_m2_mean_15drive,rent_2026_price_per_m2_median_15drive,rent_2026_ads_count_sum_15drive,buy_2026_price_per_m2_mean_15drive,buy_2026_price_per_m2_median_15drive,buy_2026_building_type_mode_15drive,buy_2026_object_type_mean_15drive,buy_2026_ads_count_sum_15drive,buy_2021_price_per_m2_mean_15drive,...,incomes_2025_over_65_num_sum_15drive,incomes_2025_men_num_sum_15drive,incomes_2025_women_num_sum_15drive,incomes_2025_with_children_num_sum_15drive,incomes_2025_with_pets_num_sum_15drive,incomes_2025_with_car_num_sum_15drive,incomes_2025_median_income_pop_mean_15drive,incomes_2025_median_income_pop_median_15drive,incomes_2025_avg_hcs_cost_pop_mean_15drive,incomes_2025_avg_hcs_cost_pop_median_15drive
0,0,2509.382185,2200.000000,4646.0,718009.753273,569526.627219,3.0,1.0,11352.0,419629.563536,...,20954.247793,256743.489536,217356.518119,112894.313981,95006.381538,195321.458886,3.608168e+07,2.271735e+07,3.152848e+06,2.023261e+06
1,1,2501.949936,2192.982456,5113.0,708760.726765,569299.476045,3.0,1.0,11993.0,414848.102537,...,22819.914865,266176.706363,232728.496507,124393.695362,101675.262556,211092.343983,4.216909e+07,2.306198e+07,3.675237e+06,2.053850e+06
2,2,2749.337227,2368.421053,4109.0,865945.056155,665859.564165,3.0,1.0,10268.0,484935.848332,...,19980.787163,205679.870055,182477.079705,99085.859371,79465.424893,177832.611613,4.489302e+07,2.495509e+07,4.005343e+06,2.180630e+06
3,3,1449.723662,1388.888889,739.0,327670.406543,329476.987143,2.0,1.0,2147.0,285557.417886,...,5607.509861,79887.819111,71518.433520,47124.881442,31577.360133,66209.520159,6.530275e+07,5.087054e+07,5.803214e+06,4.522334e+06
4,4,2618.386377,2290.525751,4008.0,762641.064328,585412.462704,3.0,1.0,10820.0,435490.283195,...,18053.546619,243762.579731,196575.205915,105122.692396,85734.924727,185035.332030,4.169766e+07,2.419067e+07,3.601425e+06,2.103061e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5024,5024,1701.065333,1564.450024,1177.0,414301.375645,389393.939394,2.0,1.0,3438.0,259874.132098,...,12127.957117,139896.074310,123812.404159,79128.466304,55037.361993,120305.758931,9.975115e+07,6.814653e+07,8.591238e+06,5.881893e+06
5025,5025,1582.622285,1519.097222,975.0,364060.592436,354782.352529,1.0,1.0,2889.0,244832.555383,...,9234.624267,102953.030560,95239.314894,60878.919907,42351.265866,89124.349370,8.963857e+07,6.791073e+07,7.419157e+06,5.734589e+06
5026,5026,1539.105091,1470.588235,663.0,362240.285337,359170.305677,1.0,1.0,2037.0,253274.147752,...,13652.638717,112489.891303,102013.989558,65091.320374,45952.313195,100436.366844,7.308563e+07,5.408366e+07,6.411638e+06,4.856602e+06
5027,5027,1500.799693,1440.365519,565.0,348507.322746,348263.401536,1.0,1.0,1510.0,250306.338692,...,12636.583248,102366.830304,94665.005581,60130.809686,42594.102896,90822.738514,7.337465e+07,5.472553e+07,6.490374e+06,5.015728e+06
